# disease-prediction-ml — GPU training (PhysioNet 2019)

Runs on Kaggle with **GPU + Internet** enabled. Clones the repo, installs it, prepares the attached dataset, trains, evaluates, and writes results into `artifacts/` (saved as the kernel Output).

In [ ]:
!nvidia-smi -L || echo 'no GPU'

In [ ]:
REPO = 'disease-prediction-ml'
import os, subprocess
if not os.path.isdir(f'/kaggle/working/{REPO}'):
    subprocess.run(['git','clone','--depth','1','https://github.com/sara-tavakoli/'+REPO+'.git'], cwd='/kaggle/working', check=True)
os.chdir(f'/kaggle/working/{REPO}')
subprocess.run(['pip','-q','install','-e','.'], check=True)
import importlib, sepsis; print('sepsis OK')

## 1 · Prepare dataset

In [ ]:
# --- stage the PhysioNet psv files into a clean root the sepsis loader understands ---
import pathlib
setA = next((p for p in pathlib.Path("/kaggle/input").rglob("training_setA") if any(p.rglob("*.psv"))), None)
assert setA, "attach a PhysioNet CinC-2019 dataset with training_setA/training_setB psv files"
src_root = setA.parent
STAGE = pathlib.Path("data/physionet"); STAGE.mkdir(parents=True, exist_ok=True)
for s in ("training_setA", "training_setB"):
    d = src_root / s
    inner = d / "training"
    target = inner if inner.is_dir() and any(inner.glob("*.psv")) else d
    link = STAGE / s
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(target.resolve())
    print(s, "->", target, "|", len(list(target.glob('*.psv'))), "stays")
ROOT = STAGE.resolve()
print("staged root:", ROOT)

## 2 · Train

In [ ]:
import subprocess
for m in ['lightgbm', 'lstm', 'gru', 'tcn', 'transformer']:
    print('=' * 20, m, '=' * 20)
    subprocess.run(['sepsis', 'train', '--config', 'configs/base.yaml',
        f'configs/model_{m}.yaml', '--set', 'data.source=physionet',
        f'data.root={ROOT}', 'data.group_by_hospital=true',
        'train.epochs=25', 'train.seed=20190804'], check=True)

## 3 · Evaluate

In [ ]:
!python scripts/update_results.py

## 4 · Show results

In [ ]:
import pathlib, IPython.display as D
for md in sorted(pathlib.Path('.').rglob('RESULTS.md')):
    D.display(D.Markdown(md.read_text()))
for png in sorted(pathlib.Path('artifacts').rglob('*.png'))[:16]:
    print(png); D.display(D.Image(str(png)))